# 🧹 Local Data Cleaning & Normalization Pipeline

This notebook runs locally on a laptop (16GB RAM, RTX 2050 GPU friendly).
It loads raw TSV files from  and , applies standardized text normalization & field cleaning, and saves the cleaned datasets to .

---
## 📦 Cell 1: Install Project Requirements

In [ ]:
import os, sys

# Locate requirements file
req_path = os.path.abspath("Error_404_Not_Found_submission/code/business_entity_resolution/requirements.txt")

if os.path.exists(req_path):
    print(f"⏳ Installing requirements from: {req_path}...")
    %pip install -r "{req_path}" --quiet
else:
    print("⏳ Installing core dependencies (pandas, numpy, rapidfuzz, xgboost, scikit-learn, pyyaml)...")
    %pip install pandas numpy rapidfuzz xgboost scikit-learn pyyaml --quiet

print("✓ All required packages installed successfully!")

---
## ⚙️ Cell 2: Environment & Module Setup

In [ ]:
import os
import sys
import gc
import time
import pandas as pd
import numpy as np
from IPython.display import display

# Add src directory to Python search path
possible_src_paths = [
    os.path.abspath("Error_404_Not_Found_submission/code/business_entity_resolution/src"),
    "/kaggle/working/ml_challenge/Error_404_Not_Found_submission/code/business_entity_resolution/src",
    os.path.abspath("src"),
]

for p in possible_src_paths:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

from cleaning import clean_source_dataframe
from io_utils import read_tsv, save_tsv

print("✓ Imports and cleaning modules initialized successfully!")
print(f"✓ Current working directory: {os.getcwd()}")

---
## 📁 Cell 3: Path & Output Directory Configuration

In [ ]:
# Base directories
BASE_DATA_DIR = os.path.abspath("dataset")
TRAIN_DIR = os.path.join(BASE_DATA_DIR, "train")
TEST_DIR = os.path.join(BASE_DATA_DIR, "test")
CLEANED_DIR = os.path.join(BASE_DATA_DIR, "cleaned")

# Create output directory for cleaned datasets
os.makedirs(CLEANED_DIR, exist_ok=True)

print("📁 Path Configuration:")
print(f"   Train Input Dir : {TRAIN_DIR}")
print(f"   Test Input Dir  : {TEST_DIR}")
print(f"   Cleaned Output  : {CLEANED_DIR}")

---
## 🛠️ Cell 4: Memory-Safe Cleaning Function (Chunked for 16GB RAM)

In [ ]:
def process_and_clean_file(input_path: str, output_path: str, source_label: str, chunksize: int = 250000) -> str:
    """
    Cleans a TSV file in memory-efficient chunks to fit comfortably within 16GB RAM.
    """
    if not os.path.exists(input_path):
        print(f"⚠️ Input file not found: {input_path}")
        return output_path

    start_time = time.time()
    print(f"
⏳ [{source_label}] Processing: {input_path}")
    
    total_rows = 0
    first_chunk = True
    
    # Read TSV in chunks to optimize memory
    for chunk in pd.read_csv(input_path, sep="\t", dtype=str, keep_default_na=False, chunksize=chunksize):
        cleaned_chunk = clean_source_dataframe(chunk, source_label=f"{source_label}_chunk")
        mode = "w" if first_chunk else "a"
        header = first_chunk
        cleaned_chunk.to_csv(output_path, sep="\t", index=False, mode=mode, header=header)
        total_rows += len(cleaned_chunk)
        first_chunk = False
        del chunk, cleaned_chunk
        gc.collect()

    elapsed = time.time() - start_time
    print(f"✅ [{source_label}] Complete — Saved {total_rows:,} rows to {output_path} ({elapsed:.2f}s)")
    return output_path

---
## 🏃 Cell 5: Clean Training Data (S1, S2, S3)

In [ ]:
# Clean Training Source 1
train_s1_in = os.path.join(TRAIN_DIR, "train_source1.tsv")
train_s1_out = os.path.join(CLEANED_DIR, "train_source1_clean.tsv")
process_and_clean_file(train_s1_in, train_s1_out, "train_S1")

# Clean Training Source 2
train_s2_in = os.path.join(TRAIN_DIR, "train_source2.tsv")
train_s2_out = os.path.join(CLEANED_DIR, "train_source2_clean.tsv")
process_and_clean_file(train_s2_in, train_s2_out, "train_S2")

# Clean Training Source 3
train_s3_in = os.path.join(TRAIN_DIR, "train_source3.tsv")
train_s3_out = os.path.join(CLEANED_DIR, "train_source3_clean.tsv")
process_and_clean_file(train_s3_in, train_s3_out, "train_S3")

---
## 🧪 Cell 6: Clean Test Data (S1, S2, S3)

In [ ]:
# Clean Test Source 1
test_s1_in = os.path.join(TEST_DIR, "test_source1.tsv")
test_s1_out = os.path.join(CLEANED_DIR, "test_source1_clean.tsv")
process_and_clean_file(test_s1_in, test_s1_out, "test_S1")

# Clean Test Source 2
test_s2_in = os.path.join(TEST_DIR, "test_source2.tsv")
test_s2_out = os.path.join(CLEANED_DIR, "test_source2_clean.tsv")
process_and_clean_file(test_s2_in, test_s2_out, "test_S2")

# Clean Test Source 3
test_s3_in = os.path.join(TEST_DIR, "test_source3.tsv")
test_s3_out = os.path.join(CLEANED_DIR, "test_source3_clean.tsv")
process_and_clean_file(test_s3_in, test_s3_out, "test_S3")

---
## 📊 Cell 7: Inspect & Validate Cleaned Output Files

In [ ]:
cleaned_files = [f for f in os.listdir(CLEANED_DIR) if f.endswith(".tsv")]
print(f"🎉 Total Cleaned TSV Files in {CLEANED_DIR}: {len(cleaned_files)}
")

for fname in sorted(cleaned_files):
    fpath = os.path.join(CLEANED_DIR, fname)
    sample_df = pd.read_csv(fpath, sep="\t", nrows=5, dtype=str, keep_default_na=False)
    print(f"📄 File: {fname}")
    print(f"   Columns ({len(sample_df.columns)}): {list(sample_df.columns)}")
    display(sample_df.head(2))
    print("-" * 80)